In [1]:
%cd ..

/home/ricka/Git/GitHub/RickArko/kaggle/playground/kcom-store-sales


# Time-Series Prediction Visualization

Visualise actual vs predicted sales for selected store-family combinations.
Loads a saved experiment run, re-runs inference, and plots time-series comparisons.

In [2]:
from __future__ import annotations

import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from store_sales.data import load_config, load_data, merge_tables
from store_sales.features import TimeSeriesFeatureEngineer
from store_sales.metrics import rmsle
from store_sales.models import TimeSeriesModel

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

## 1. Configuration

Point to one or more completed run directories. The notebook loads the frozen config and trained model from each run.

In [3]:
# ── CONFIGURE HERE ────────────────────────────────────────────────
# Leave empty to auto-discover the best full-dataset runs (one per model type).
RUN_DIRS: list[str] = []
# Or pin specific runs:
# RUN_DIRS = [
#     "outputs/runs/20260707_093203_bench-log-ridge",
#     "outputs/runs/20260710_122230_bench-lightgbm",
# ]
# ──────────────────────────────────────────────────────────────────

from store_sales.viz import discover_runs

if RUN_DIRS:
    RUN_DIRS = [Path(d) for d in RUN_DIRS]
    for d in RUN_DIRS:
        assert d.exists(), f"Run directory not found: {d}"
else:
    discovered = discover_runs(Path("outputs/runs"), scope="full", prefer=["log-ridge", "lightgbm", "nixtla"])
    if not discovered:
        raise FileNotFoundError(
            "No completed runs found. Train first:\n"
            "  make train-linear CONFIG=config/linear-log.yaml RUN_NAME=demo-log-ridge\n"
            "  make train CONFIG=config/baseline.yaml RUN_NAME=demo-lightgbm"
        )
    RUN_DIRS = discovered

# How many (store, family) series to plot in the per-series view
N_SERIES = 6

print(f"Runs ({len(RUN_DIRS)}):")
for d in RUN_DIRS:
    cfg = load_config(str(d / "config.yaml"))
    print(f"  {d.name}  ({cfg['model']['type']})")

Runs (2):
  20260707_093203_bench-log-ridge  (linear)
  20260710_122230_bench-lightgbm  (lightgbm)


## 2. Load data & feature engineer

Shared data loading. Each model run may use slightly different features; we build features per run below.

In [4]:
tables = load_data()
train, test = merge_tables(tables)
target_col = "sales"
y_full = train[target_col].copy()
print(f"Train: {train.shape}  Test: {test.shape}")

Train: (3000888, 13)  Test: (28512, 12)


In [5]:
def predict_run(run_dir, train, test, y_full):
    """Load run, engineer features, predict, return (date, store, family, actual, predicted, split)."""
    cfg = load_config(str(run_dir / "config.yaml"))
    feat_cfg = cfg["features"]
    target_col = cfg["competition"]["target"]
    target_transform = cfg.get("model", {}).get("target_transform", "raw")

    run_train = train.copy()
    run_test = test.copy()
    if target_transform == "log1p":
        run_train["log_sales"] = np.log1p(run_train[target_col])
        run_test["log_sales"] = 0.0

    engineer = TimeSeriesFeatureEngineer(
        date_col=feat_cfg.get("date_col", "date"),
        store_col=feat_cfg.get("store_col", "store_nbr"),
        family_col=feat_cfg.get("family_col", "family"),
        onpromotion_col=feat_cfg.get("onpromotion_col", "onpromotion"),
        date_features=feat_cfg.get("date_features", []),
        drop_cols=feat_cfg.get("drop_cols", []),
        lag_config=feat_cfg.get("lag_features", []),
        rolling_config=feat_cfg.get("rolling_features", []),
        fourier_config=feat_cfg.get("fourier_features"),
        ref_date=run_train["date"].min(),
    )

    X_lag, X_test_feat = engineer.create_lag_features(run_train, run_test, target_col)
    engineer.fit(X_lag)
    X_all = engineer.transform(X_lag)
    X_test = engineer.transform(X_test_feat)

    # One-hot encode if needed (detect from model's feature_names_)
    ts_model = TimeSeriesModel.load(run_dir / "models" / "model.joblib")
    model_feats = set(ts_model.feature_names_)
    cat_cols = [c for c in X_all.columns if X_all[c].dtype.name == "category"]
    needs_ohe = bool(cat_cols) and not (cat_cols and cat_cols[0] in model_feats)

    if needs_ohe:
        known_cats = {c: sorted(X_all[c].cat.categories.tolist()) for c in cat_cols}
        for df in (X_all, X_test):
            for c in cat_cols:
                df[c] = pd.Categorical(df[c], categories=known_cats[c])
        X_all = pd.get_dummies(X_all, columns=cat_cols, drop_first=True, dtype=int)
        X_test = pd.get_dummies(X_test, columns=cat_cols, drop_first=True, dtype=int)
        train_cols = list(X_all.columns)
        for c in set(train_cols) - set(X_test.columns):
            X_test[c] = 0
        X_test = X_test[train_cols]

    train_preds = ts_model.predict(X_all)
    test_preds = ts_model.predict(X_test)
    if target_transform == "log1p":
        train_preds = np.expm1(train_preds)
        test_preds = np.expm1(test_preds)
    train_preds = np.maximum(train_preds, 0)
    test_preds = np.maximum(test_preds, 0)

    meta = X_lag[["date", "store_nbr", "family"]].copy()
    meta["actual"] = y_full.loc[X_lag.index].values
    meta["predicted"] = train_preds
    meta["split"] = "train"

    meta_test = X_test_feat[["date", "store_nbr", "family"]].copy()
    meta_test["actual"] = np.nan
    meta_test["predicted"] = test_preds
    meta_test["split"] = "test"

    return pd.concat([meta, meta_test], ignore_index=True)


# Predict for each run
results = {}
for run_dir in RUN_DIRS:
    name = run_dir.name
    print(f"Predicting {name} ...")
    results[name] = predict_run(run_dir, train, test, y_full)

print("Done.")

Predicting 20260707_093203_bench-log-ridge ...


KeyError: 'Column not found: log_sales'

## 3. Daily aggregate: actual vs predicted by model

Sum sales across all stores for each day. Shows the big-picture fit.
Only validation-period days are shown (where both actual and predicted exist).

In [ ]:
# Determine validation period
all_dates = sorted(train["date"].unique())
val_split_date = all_dates[-16]
val_dates_set = set(all_dates[-16:])

fig, ax = plt.subplots(figsize=(16, 5))

# Plot actuals (from first model — all share the same train data)
first_name = list(results.keys())[0]
first_df = results[first_name]
daily_actual = (
    first_df[first_df["split"] == "train"]
    .groupby("date")["actual"]
    .sum()
    .reset_index()
)
daily_actual["date"] = pd.to_datetime(daily_actual["date"])
val_actual = daily_actual[daily_actual["date"].isin(val_dates_set)].dropna()
ax.plot(val_actual["date"], val_actual["actual"], color="black", linewidth=1.0,
        marker=".", markersize=4, label="Actual")

# Plot predictions per model
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]
for idx, (name, df) in enumerate(results.items()):
    daily_pred = (
        df[df["split"] == "train"]
        .groupby("date")["predicted"]
        .sum()
        .reset_index()
    )
    daily_pred["date"] = pd.to_datetime(daily_pred["date"])
    val_pred = daily_pred[daily_pred["date"].isin(val_dates_set)].dropna()

    # Compute RMSLE on completed predictions only (val period)
    merged = val_actual.merge(val_pred, on="date", suffixes=("_a", "_p"))
    score = rmsle(merged["actual"].values, merged["predicted"].values)

    ax.plot(val_pred["date"], val_pred["predicted"], color=colors[idx % len(colors)],
            linewidth=0.9, marker=".", markersize=4,
            label=f"{name}  (val RMSLE={score:.4f})")

ax.set_title("Daily aggregate sales — actual vs predicted (validation period only)",
             fontsize=13, fontweight="bold")
ax.set_ylabel("Total sales (sum across all stores)")
ax.legend(fontsize=8, ncol=1, loc="upper right")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
plt.show()

## 4. Per-series RMSLE

Compute validation RMSLE for each (store, family) combination.

In [ ]:
# Use the first model's predictions for per-series analysis
run_name = list(results.keys())[0]
combined = results[run_name]

val_meta = combined[
    (combined["split"] == "train") & (combined["date"].isin(val_dates_set))
].copy()

series_rmsle = []
for (store, family), grp in val_meta.groupby(["store_nbr", "family"]):
    y_true = grp["actual"].values
    y_pred = grp["predicted"].values
    series_rmsle.append({
        "store_nbr": store,
        "family": family,
        "series": f"{store}_{family}",
        "val_rmsle": round(rmsle(y_true, y_pred), 4),
        "mean_actual": round(y_true.mean(), 1),
        "mean_pred": round(y_pred.mean(), 1),
    })

rmsle_df = pd.DataFrame(series_rmsle).sort_values("val_rmsle", ascending=False)
print(f"Per-series val RMSLE: min={rmsle_df['val_rmsle'].min():.4f}, "
      f"median={rmsle_df['val_rmsle'].median():.4f}, "
      f"max={rmsle_df['val_rmsle'].max():.4f}")
print()
print("Worst 5:")
display(rmsle_df.head(5))
print()
print("Best 5:")
display(rmsle_df.tail(5))

## 5. Pick series and plot

Select a few interesting store-family combinations to visualize.

In [ ]:
combined["series"] = combined["store_nbr"].astype(str) + "_" + combined["family"].astype(str)

# Pick the N_SERIES with the highest total sales (most interesting visually)
series_total = (
    combined[combined["split"] == "train"]
    .groupby("series")["actual"]
    .sum()
    .sort_values(ascending=False)
)
pick_series = series_total.head(N_SERIES).index.tolist()

print("Selected series (top by total sales):")
for s in pick_series:
    print(f"  {s}")

In [ ]:
def plot_series(
    data: pd.DataFrame,
    series_name: str,
    val_split_date: str | None = None,
    days_back: int = 180,
):
    """Plot actual vs predicted for one store-family series."""
    sdata = data[data["series"] == series_name].copy()
    sdata = sdata.sort_values("date")

    if days_back:
        last_date = sdata["date"].max()
        cutoff = last_date - pd.Timedelta(days=days_back)
        sdata = sdata[sdata["date"] >= cutoff]

    fig, ax = plt.subplots(figsize=(14, 4.5))

    train_part = sdata[sdata["split"] == "train"]
    test_part = sdata[sdata["split"] == "test"]

    if val_split_date and len(train_part) > 0:
        val_part = train_part[train_part["date"] >= val_split_date].copy()
        train_part = train_part[train_part["date"] < val_split_date].copy()
        val_part["split"] = "val"
        val_actuals = val_part.dropna(subset=["actual"])
        val_preds_only = val_part[["date", "predicted", "split"]].dropna()
    else:
        val_actuals = pd.DataFrame()
        val_preds_only = pd.DataFrame()

    ax.plot(
        train_part["date"], train_part["actual"],
        color="#aaaaaa", linewidth=0.5, alpha=0.6, label="Train actual",
    )

    if len(val_actuals) > 0:
        ax.plot(
            val_actuals["date"], val_actuals["actual"],
            color="#1f77b4", linewidth=1.2, label="Val actual",
        )

    if len(val_preds_only) > 0:
        ax.plot(
            val_preds_only["date"], val_preds_only["predicted"],
            color="#ff7f0e", linewidth=1.2, linestyle="--", label="Val predicted",
        )

    if len(test_part) > 0:
        ax.plot(
            test_part["date"], test_part["predicted"],
            color="#d62728", linewidth=1.5, linestyle="--", marker="o",
            markersize=3, label="Test forecast",
        )

    if val_split_date:
        ax.axvline(pd.Timestamp(val_split_date), color="black", linewidth=0.8, linestyle=":", alpha=0.5)
        ax.text(pd.Timestamp(val_split_date), ax.get_ylim()[1] * 0.95,
                "train/val split", fontsize=8, rotation=90, va="top", alpha=0.6)

    store_nbr, family = series_name.split("_", 1)
    ax.set_title(f"Store {store_nbr} — {family}", fontsize=12, fontweight="bold")
    ax.set_ylabel("Sales")
    ax.legend(fontsize=8, ncol=3, loc="upper left")
    ax.tick_params(axis="x", rotation=30)
    fig.tight_layout()
    plt.show()


split_date = str(val_split_date)
print(f"Validation split date: {split_date}")

In [ ]:
# ---- Plot selected series ----
print(f"Plotting {len(pick_series)} series (showing last 180 days each)...")
for series_name in pick_series:
    plot_series(combined, series_name, val_split_date=split_date, days_back=180)

## 6. Plot worst-performing series

Also visualize the store-family combinations with the highest RMSLE to spot failure modes.

In [ ]:
worst_series = rmsle_df.head(N_SERIES)["series"].tolist()
print(f"Plotting {len(worst_series)} worst-performing series...")
for series_name in worst_series:
    plot_series(combined, series_name, val_split_date=split_date, days_back=180)

## 7. Distribution of per-series RMSLE

Histogram showing how RMSLE varies across store-family combinations.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.hist(rmsle_df["val_rmsle"], bins=60, color="#1f77b4", edgecolor="white", linewidth=0.3)
ax.axvline(rmsle_df["val_rmsle"].median(), color="red", linestyle="--",
           label=f"Median {rmsle_df['val_rmsle'].median():.3f}")
ax.axvline(rmsle_df["val_rmsle"].mean(), color="orange", linestyle="--",
           label=f"Mean {rmsle_df['val_rmsle'].mean():.3f}")
ax.set_xlabel("Val RMSLE")
ax.set_ylabel("Series count")
ax.set_title("Distribution of per-series validation RMSLE")
ax.legend(fontsize=8)

ax = axes[1]
ax.scatter(rmsle_df["mean_actual"], rmsle_df["val_rmsle"], s=8, alpha=0.4, color="#1f77b4")
ax.set_xlabel("Mean actual sales")
ax.set_ylabel("Val RMSLE")
ax.set_title("RMSLE vs. mean sales")
ax.set_xscale("symlog")

fig.tight_layout()
plt.show()

## 8. Overall metrics

In [ ]:
print("Per-model validation RMSLE (daily aggregate):")
for name, df in results.items():
    daily_pred = df[df["split"] == "train"].groupby("date")["predicted"].sum().reset_index()
    daily_pred["date"] = pd.to_datetime(daily_pred["date"])
    val_pred = daily_pred[daily_pred["date"].isin(val_dates_set)].dropna()
    merged = val_actual.merge(val_pred, on="date", suffixes=("_a", "_p"))
    score = rmsle(merged["actual"].values, merged["predicted"].values)
    model_type = load_config(str(Path(name).parent / name / "config.yaml"))["model"]["type"] if False else name
    print(f"  {name}: {score:.4f}")

## 9. TOTO Foundation Model Comparison

Run TOTO zero-shot forecasting and compare relative errors vs. other models.
TOTO (Datadog's Time-series Optimized Transformer for Observability) is a
foundation model pre-trained on billions of time series data points.


In [ ]:
from store_sales.toto_pipeline import TotoPipeline

# Run TOTO zero-shot forecasting
toto_pipe = TotoPipeline(
    model_name="Datadog/Toto-2.0-22m",
    decode_block_size=768,
)
toto_sub = toto_pipe.predict(train, test, horizon=16)
toto_val_score = toto_pipe.validate(train, horizon=16, val_days=16)
print(f"\nTOTO validation RMSLE: {toto_val_score:.6f}")

# Build per-series TOTO predictions for val period
val_dates = sorted(train["date"].unique())[-16:]
val_set = set(val_dates)
train_until = sorted(train["date"].unique())[-32]
train_trunc = train[train["date"] < train_until].copy()
val_actuals_raw = train[train["date"].isin(val_set)].copy()
toto_val_sub = toto_pipe.predict(train_trunc, val_actuals_raw, horizon=16)

# Compute per-series RMSLE for TOTO
val_actuals_raw["_series"] = (
    val_actuals_raw["store_nbr"].astype(str) + "_" + val_actuals_raw["family"].astype(str)
)
toto_val_sub["_series"] = (
    val_actuals_raw.set_index("id").loc[toto_val_sub["id"], "store_nbr"].astype(str)
    + "_" + val_actuals_raw.set_index("id").loc[toto_val_sub["id"], "family"].astype(str)
)
toto_val_sub["date"] = toto_val_sub["id"].map(
    val_actuals_raw.set_index("id")["date"]
)

toto_per_series = []
for (store, family), grp in val_actuals_raw.groupby(["store_nbr", "family"]):
    series_name = f"{store}_{family}"
    y_true = grp["sales"].values
    pred_grp = toto_val_sub[toto_val_sub["_series"] == series_name]
    if len(pred_grp) == 0:
        continue
    y_pred = pred_grp.sort_values("date")["sales"].values
    min_len = min(len(y_true), len(y_pred))
    if min_len == 0:
        continue
    y_true, y_pred = y_true[:min_len], y_pred[:min_len]
    toto_per_series.append({
        "store_nbr": store,
        "family": family,
        "series": series_name,
        "val_rmsle": round(rmsle(y_true, y_pred), 4),
    })
toto_rmsle_df = pd.DataFrame(toto_per_series).sort_values("val_rmsle", ascending=False)
print(f"TOTO per-series val RMSLE: median={toto_rmsle_df["val_rmsle"].median():.4f}")


## 10. TOTO Relative Error vs. Best Other Model

For each store-family series, show whether TOTO outperforms or underperforms
the best non-TOTO model. Positive values mean TOTO has **lower** RMSLE (better).


In [ ]:
# Get per-series RMSLE from each non-TOTO model, pick the best per series
ref_name = list(results.keys())[0]
ref_df = results[ref_name]
ref_per_series = []
for (store, family), grp in ref_df[
    (ref_df["split"] == "train") & (ref_df["date"].isin(val_set))
].groupby(["store_nbr", "family"]):
    y_true = grp["actual"].values
    y_pred = grp["predicted"].values
    ref_per_series.append({
        "series": f"{store}_{family}",
        f"{ref_name}_rmsle": round(rmsle(y_true, y_pred), 4),
    })
ref_rmsle_df = pd.DataFrame(ref_per_series)

# Merge TOTO vs reference
compare = toto_rmsle_df.merge(ref_rmsle_df, on="series", how="inner")
compare["toto_better"] = compare["val_rmsle"] < compare[f"{ref_name}_rmsle"]
compare["relative_improvement"] = (
    (compare[f"{ref_name}_rmsle"] - compare["val_rmsle"]) / compare[f"{ref_name}_rmsle"]
) * 100

n_better = compare["toto_better"].sum()
n_total = len(compare)
print(f"TOTO beats {ref_name} on {n_better}/{n_total} series ({100*n_better/n_total:.1f}%)")
print(f"Median relative improvement: {compare['relative_improvement'].median():+.2f}%")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: scatter of per-series RMSLE
ax = axes[0]
max_rmsle = max(compare["val_rmsle"].max(), compare[f"{ref_name}_rmsle"].max())
ax.plot([0, max_rmsle], [0, max_rmsle], "k--", linewidth=0.8, alpha=0.4)
colors_comp = ["#2ca02c" if b else "#d62728" for b in compare["toto_better"]]
ax.scatter(compare[f"{ref_name}_rmsle"], compare["val_rmsle"],
           c=colors_comp, s=8, alpha=0.6)
ax.set_xlabel(f"{ref_name} per-series RMSLE")
ax.set_ylabel("TOTO per-series RMSLE")
ax.set_title("TOTO vs Reference: per-series RMSLE")

# Right: histogram of relative improvement
ax = axes[1]
ax.hist(compare["relative_improvement"], bins=50, color="#1f77b4", alpha=0.7,
        edgecolor="white", linewidth=0.3)
ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xlabel("Relative improvement (%) \u2014 positive = TOTO better")
ax.set_ylabel("Number of series")
ax.set_title("Distribution of TOTO relative improvement")

fig.tight_layout()
plt.show()


## 11. Multi-Model Comparison Table

Compare all available models side-by-side.


In [ ]:
comparison_rows = []

# All non-TOTO models
for name, df in results.items():
    daily_pred = df[df["split"] == "train"].groupby("date")["predicted"].sum().reset_index()
    daily_pred["date"] = pd.to_datetime(daily_pred["date"])
    val_pred = daily_pred[daily_pred["date"].isin(val_set)].dropna()
    merged = val_actual.merge(val_pred, on="date", suffixes=("_a", "_p"))
    score = rmsle(merged["actual"].values, merged["predicted"].values)
    try:
        cfg = load_config(str(Path(name).parent / name / "config.yaml"))
        model_type = cfg.get("model", {}).get("type", "?")
    except:
        model_type = "?"
    comparison_rows.append({
        "model": name,
        "type": model_type,
        "val_rmsle_daily": round(score, 4),
        "val_rmsle_avg_series": round(rmsle_df["val_rmsle"].mean(), 4),
    })

# TOTO
comparison_rows.append({
    "model": "TOTO-22m",
    "type": "toto",
    "val_rmsle_daily": round(toto_val_score, 4),
    "val_rmsle_avg_series": round(toto_rmsle_df["val_rmsle"].mean(), 4),
})

comparison_df = pd.DataFrame(comparison_rows).sort_values("val_rmsle_daily")
comparison_df["delta_vs_best"] = (
    comparison_df["val_rmsle_daily"] - comparison_df["val_rmsle_daily"].min()
).round(4)

print("=" * 70)
print("  Model Comparison \u2014 validation RMSLE (lower is better)")
print("=" * 70)
print(comparison_df.to_string(index=False))
print("=" * 70)
